# Qwen3.5-4B → OpenVINO IR + HuggingFace Push

Runs the scripts in `qwen35/scripts/` end-to-end, exactly the same way
you would run them locally from a terminal.

| Step | Script called |
|------|---------------|
| 0 | Load Kaggle secrets |
| 1 | `git clone dsainvg/openvino-model-conv` |
| 2 | Install requirements |
| 3 | `pytest qwen35/tests/` *(toy smoke test, no weights needed)* |
| 4 | `qwen35/scripts/download_model.py` — pull weights from HF |
| 5 | `qwen35/scripts/convert_to_openvino.py` — ov.convert_model → IR |
| 6 | `qwen35/scripts/push_to_hf.py` — create HF repo + upload |

> **Kaggle Secrets needed** (Add-ons → Secrets):
> - `HF_TOKEN` — HuggingFace token with **write** scope
> - `HF_REPO_NAME` — target repo name *(default: `qwen35-4b-openvino-fp16`)*


## 0 · Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(key, fallback=None):
        try:
            return _s.get_secret(key)
        except Exception:
            return fallback
except ImportError:
    def _get(key, fallback=None):
        return os.environ.get(key, fallback)

HF_TOKEN     = _get("HF_TOKEN")
HF_REPO_NAME = _get("HF_REPO_NAME", "qwen35-4b-openvino-fp16")

if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN secret is missing. Add it under Add-ons → Secrets.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"HF_REPO_NAME : {HF_REPO_NAME}")
print("HF_TOKEN     : *** (set)")

## 1 · Clone the converter repo

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/dsainvg/openvino-model-conv.git"
REPO_DIR = Path("/kaggle/working/openvino-model-conv")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)

QWEN35_DIR  = REPO_DIR / "qwen35"
SCRIPTS_DIR = QWEN35_DIR / "scripts"
MODEL_DIR   = Path("/kaggle/working/Qwen3.5-4B")
OUTPUT_DIR  = Path("/kaggle/working/ov_ir_qwen35_4b")

print(f"Repo    : {REPO_DIR}")
print(f"Scripts : {SCRIPTS_DIR}")

## 2 · Install requirements

In [ ]:
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("transformers", "openvino", "huggingface_hub", "safetensors",
    "sentencepiece", "tiktoken", "accelerate", "pytest")

print("Done.")

## 3 · Toy smoke test  (`qwen35/tests/`)

Runs against random-weight toy models — no downloads needed. Should finish in seconds.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("Toy smoke tests FAILED — fix modeling code before converting real weights.")
print("\n✓ All toy tests passed.")

## 4 · Download Qwen3.5-4B weights  (`qwen35/scripts/download_model.py`)

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "download_model.py"),
        "--model",  "Qwen/Qwen3.5-4B",
        "--output", str(MODEL_DIR),
        "--token",  HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("download_model.py failed.")
print("\n✓ Model downloaded.")

## 5 · Convert to OpenVINO IR  (`qwen35/scripts/convert_to_openvino.py`)

Loads real BF16 weights, traces with `ov.convert_model`, saves `openvino_model.xml/.bin`.

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "convert_to_openvino.py"),
        "--model-dir", str(MODEL_DIR),
        "--output",    str(OUTPUT_DIR),
        "--dtype",     "bf16",
        # --compile-check omitted: Kaggle CPU may lack Intel-specific OV plugins
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("convert_to_openvino.py failed.")
print("\n✓ IR saved to", OUTPUT_DIR)

## 6 · Push to HuggingFace  (`qwen35/scripts/push_to_hf.py`)

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "push_to_hf.py"),
        "--ir-dir",    str(OUTPUT_DIR),
        "--repo-name", HF_REPO_NAME,
        "--token",     HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("push_to_hf.py failed.")